# Tips 02：複数のExcelファイルを結合

**業務の困りごと**：店舗ごとの売上表をコピーしてまとめる。

**このTipsでできること**：`glob("*.xlsx")` で対象ファイルを集め、各ファイル名から店舗名を取り出します。`concat` は同じ列構成の表を縦方向に結合します。

**使用ライブラリ**：pandas, pathlib

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

鹿児島・霧島・姶良の3店舗について、商品と金額を記録したExcelを1つずつ作ります。

最後の `head()` で先頭の行を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
import pandas as pd
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 店舗ごとの入力ファイルを置くフォルダを作る
folder=base/'tips02_店舗別'; folder.mkdir(exist_ok=True)
# 3店舗分のExcelをそれぞれ作る
for shop,amount in [('鹿児島',1200),('霧島',900),('姶良',1100)]:
    pd.DataFrame({'商品':['茶','菓子'],'売上金額':[amount,500]}).to_excel(folder/f'{shop}.xlsx',index=False)
print(sorted(p.name for p in folder.glob('*.xlsx')))
print('作成した店舗ファイル（先頭5件）:')
display(pd.DataFrame({'ファイル名':[p.name for p in sorted(folder.glob('*.xlsx'))]}).head())
print('鹿児島店のデータ（先頭5行）:')
pd.read_excel(folder/'鹿児島.xlsx').head()


## 2. 自動処理

`glob("*.xlsx")` で対象ファイルを集め、各ファイル名から店舗名を取り出します。`concat` は同じ列構成の表を縦方向に結合します。

**結果を見るポイント**：結合後は **6行・3店舗・合計4,700円** です。ファイル名と表の行数を変えたとき、想定外のファイルまで取り込まないか確認しましょう。


In [ ]:
# 読み込んだ店舗別の表を入れるリスト
parts=[]
# ファイル名順に1つずつ読み込む
for path in sorted(folder.glob('*.xlsx')):
    frame=pd.read_excel(path)
    # 後から出所が分かるよう店舗名の列を追加する
    frame['店舗']=path.stem
    parts.append(frame)
# 縦方向に結合し行番号を振り直す
merged=pd.concat(parts,ignore_index=True)
out=base/'Tips02_全店舗.xlsx'; merged.to_excel(out,index=False)
print(merged)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

結合後は **6行・3店舗・合計4,700円** です。ファイル名と表の行数を変えたとき、想定外のファイルまで取り込まないか確認しましょう。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert len(merged)==6 and merged['店舗'].nunique()==3
assert merged['売上金額'].sum()==4700
print('確認OK:',out)


## 4. 持ち帰り・練習

**練習問題**：同じフォルダに4店舗目を追加して再実行しましょう。

入力フォルダには結合対象のExcelだけを置きます。出力は別の場所に保存して再読込を防ぎます。

**実務で使う前に**：店舗ごとのExcelは同じ列名を想定します。実務では入力フォルダに対象ファイルだけを置いてください。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
